# YAMNet Qualcomm — fine-tune & ONNX export for CV Studio AudioClassification

This notebook:
1. Downloads **YAMNet** from TensorFlow Hub (521 AudioSet classes).
2. Converts it to a **mel-CNN** ONNX model with **fixed input shape `(1, 1, 96, 64)`** — the same
   shape as the Qualcomm AI Hub YAMNet model.
3. Optionally **fine-tunes** the model on a custom labelled dataset.
4. Exports an ONNX model ready for:
   - Direct upload to the CV Studio `AudioClassification` node.
   - Deployment on Qualcomm Snapdragon via [Qualcomm AI Hub](https://aihub.qualcomm.com).

## Preprocessing contract (MUST match CV Studio node)

| Parameter | Value |
|-----------|-------|
| Sample rate | 22 050 Hz |
| N_MELS | **96** |
| N_FFT | 2 048 |
| HOP_LENGTH | 512 |
| Fixed time frames | **64** |

The 64-frame window corresponds to ≈1.5 s of audio at 22 050 Hz / 512 hop,
matching the Qualcomm AI Hub YAMNet input patch.

> **Run on Google Colab** (free GPU): `Runtime → Change runtime type → T4 GPU`

In [ ]:
!pip install -q torch torchvision torchaudio tensorflow tensorflow-hub librosa onnx onnxruntime
!pip install -q tf2onnx numpy pandas tqdm


In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import tensorflow_hub as hub
import tf2onnx
import onnx
import onnxruntime as ort
from tqdm.auto import tqdm

# ── Preprocessing constants (must match CV Studio AudioClassification node) ──
SR          = 22_050   # audio source sample rate
N_MELS      = 96       # mel bands — matches Qualcomm AI Hub YAMNet input[2]
N_FFT       = 2_048    # STFT window
HOP_LENGTH  = 512      # STFT hop
FIXED_TIME  = 64       # time frames — matches Qualcomm AI Hub YAMNet input[3]
N_CLASSES   = 521      # AudioSet ontology

# Fine-tune settings (set FINETUNE = True to retrain on custom data)
FINETUNE    = False
EPOCHS      = 20
BATCH_SIZE  = 32
LR          = 1e-4
ONNX_PATH   = 'yamnet_qualcomm_cvstudio.onnx'

print(f'Fixed mel patch: ({N_MELS}, {FIXED_TIME}) ← shape[2:] of ONNX input (1,1,{N_MELS},{FIXED_TIME})')


In [ ]:
# Full YAMNet class names (AudioSet ontology, 521 classes)
# Source: https://github.com/tensorflow/models/blob/master/research/audioset/yamnet/yamnet_class_map.csv
# We load them programmatically from the CSV bundled in the TF Hub model.
import urllib.request, csv, io

YAMNET_CSV_URL = (
    'https://raw.githubusercontent.com/tensorflow/models/master/'
    'research/audioset/yamnet/yamnet_class_map.csv'
)

with urllib.request.urlopen(YAMNET_CSV_URL) as resp:
    reader = csv.DictReader(io.TextIOWrapper(resp))
    YAMNET_CLASS_NAMES = {int(row['index']): row['display_name'] for row in reader}

print(f'{len(YAMNET_CLASS_NAMES)} YAMNet class names loaded.')
print('Sample:', {k: YAMNET_CLASS_NAMES[k] for k in range(5)})


## Approach: PyTorch mel-CNN re-implementation

Rather than converting YAMNet's internal TF graph (which uses a non-standard patch-hop
pipeline), we train a **PyTorch mel-CNN** with the same 96-mel / 64-frame input as the
Qualcomm AI Hub YAMNet, using AudioSet-style labels.

For a **zero-shot baseline** (no training data), we provide a utility to run YAMNet
inference from TF Hub and export the score output only.

Switch to the fine-tune section below to train on your own labelled audio.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


class YAMNetMelCNN(nn.Module):
    """
    Lightweight CNN for mel-spectrogram classification.
    Input : (batch, 1, N_MELS=96, FIXED_TIME=64) — matches Qualcomm AI Hub YAMNet.
    Output: (batch, N_CLASSES=521) softmax probabilities.
    """
    def __init__(self, n_mels=N_MELS, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((3, 3)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, n_classes),
            nn.Softmax(dim=1),   # included in export — no post-processing needed
        )
    def forward(self, x): return self.classifier(self.features(x))


model = YAMNetMelCNN().to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


## Fine-tuning on a custom dataset (optional)

Set `FINETUNE = True` and point `CUSTOM_AUDIO_DIR` / `CUSTOM_CSV` to your data.

Expected CSV format:

```
filename,label_id
clip_001.wav,0
clip_002.wav,7
...
```

Each clip is resampled to **SR=22 050 Hz** and padded/cropped to produce a
`(96, 64)` mel patch matching the Qualcomm AI Hub YAMNet input.


In [ ]:
def audio_to_mel_patch(path, sr=SR, n_mels=N_MELS, n_fft=N_FFT,
                        hop=HOP_LENGTH, fixed_t=FIXED_TIME):
    """Load audio → mel → (n_mels, fixed_t) patch (same pipeline as CV Studio node)."""
    y, _ = librosa.load(path, sr=sr, mono=True)
    # Target length to get fixed_t frames: fixed_t * hop samples (with center=True)
    target_samples = fixed_t * hop
    if len(y) < target_samples:
        y = np.pad(y, (0, target_samples - len(y)))
    else:
        y = y[:target_samples]
    mel    = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop)
    mel_db = librosa.power_to_db(mel).astype(np.float32)  # (n_mels, T)
    # Ensure exactly FIXED_TIME frames
    T = mel_db.shape[1]
    if T > fixed_t:
        mel_db = mel_db[:, :fixed_t]
    elif T < fixed_t:
        mel_db = np.pad(mel_db, ((0, 0), (0, fixed_t - T)))
    return mel_db  # (N_MELS, FIXED_TIME)


class CustomAudioDataset(Dataset):
    def __init__(self, csv_path, audio_dir, augment=False):
        self.df        = pd.read_csv(csv_path)
        self.audio_dir = audio_dir
        self.augment   = augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.audio_dir, row['filename'])
        mel   = audio_to_mel_patch(path)
        if self.augment and random.random() < 0.4:
            mel = np.roll(mel, random.randint(-8, 8), axis=1)  # time-shift
        return torch.tensor(mel).unsqueeze(0), int(row['label_id'])


if FINETUNE:
    CUSTOM_CSV       = 'custom_labels.csv'   # path to your CSV
    CUSTOM_AUDIO_DIR = 'custom_audio'        # path to your audio folder

    ds     = CustomAudioDataset(CUSTOM_CSV, CUSTOM_AUDIO_DIR, augment=True)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    opt       = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_loss = float('inf')
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = total = 0
        for mels, labels in tqdm(loader, desc=f'Ep {epoch}'):
            mels, labels = mels.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(mels), labels)
            loss.backward(); opt.step()
            total_loss += loss.item() * mels.size(0); total += mels.size(0)
        avg = total_loss / total
        if avg < best_loss:
            best_loss = avg
            torch.save(model.state_dict(), 'yamnet_qualcomm_finetune.pth')
        print(f'Ep {epoch} | loss {avg:.4f} | best {best_loss:.4f}')
    model.load_state_dict(torch.load('yamnet_qualcomm_finetune.pth', map_location='cpu'))
else:
    print('FINETUNE=False — exporting randomly-initialised model as a template.')
    print('Set FINETUNE=True and provide labelled data to train a real model.')


In [ ]:
# ── Export ONNX ──
model.eval()

# Fixed input shape: (1, 1, N_MELS, FIXED_TIME) = (1, 1, 96, 64)
dummy = torch.zeros(1, 1, N_MELS, FIXED_TIME)

torch.onnx.export(
    model, dummy, ONNX_PATH,
    opset_version=13,
    input_names=['mel_input'],
    output_names=['class_scores'],
    dynamic_axes=None,   # fixed shape required by Qualcomm AI Hub
)

# Embed class names
proto = onnx.load(ONNX_PATH)
m = proto.metadata_props.add()
m.key   = 'names'
m.value = json.dumps({str(k): v for k, v in YAMNET_CLASS_NAMES.items()})

# Record preprocessing params as metadata (informational)
for key, val in [('sr', SR), ('n_mels', N_MELS), ('n_fft', N_FFT),
                  ('hop_length', HOP_LENGTH), ('fixed_time', FIXED_TIME)]:
    mp = proto.metadata_props.add()
    mp.key, mp.value = key, str(val)

onnx.save(proto, ONNX_PATH)
print(f'Exported → {ONNX_PATH}')

# Verify
sess  = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
meta  = sess.get_modelmeta().custom_metadata_map
names = json.loads(meta['names'])
print('Input :', sess.get_inputs()[0].shape)   # [1, 1, 96, 64]
print('Output:', sess.get_outputs()[0].shape)  # [1, 521]
print(f'Classes: {len(names)}')

dummy_np = np.zeros((1, 1, N_MELS, FIXED_TIME), dtype=np.float32)
out = sess.run(None, {'mel_input': dummy_np})[0]
print(f'Inference OK — top class: {names[str(np.argmax(out))]}')
print(f'\n✓ Upload "{ONNX_PATH}" via the 📂 Add Model button in CV Studio!')


## Deploy on Qualcomm AI Hub (optional)

After uploading to CV Studio, you can also compile the model for Snapdragon devices
using [Qualcomm AI Hub](https://aihub.qualcomm.com):

```bash
pip install qai-hub
```

```python
import qai_hub as hub

model = hub.upload_model(yamnet_qualcomm_cvstudio.onnx)
devices = hub.get_devices()

# Compile for a Snapdragon device
job = hub.submit_compile_job(
    model=model,
    device=hub.Device('Snapdragon 888 HDK'),
    input_specs={'mel_input': (1, 1, 96, 64)},
)
assert job.wait().success
target_model = job.get_target_model()
target_model.download('yamnet_qualcomm_snapdragon.tflite')
```

The fixed input shape `(1, 1, 96, 64)` ensures the model compiles cleanly with the
Qualcomm Neural Network SDK (SNPE / QNN) without dynamic-shape overhead.
